In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import joblib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Example/Example/1/Lung_Model") #Lung_Model path #change the path inside

# Load model
model = AutoModelForSequenceClassification.from_pretrained("Example/Example/1/1/Lung_Model") #Lung_Model path # change the path inside

#Load encoder
encoder = joblib.load("Example/Example/1/Lung_Model/label_encoder.pkl") # label_encoder.pk1 path #Change the path inside

# Move model to GPU (if available)
#model_loaded.to(device)

#model_loaded.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
import torch

def predict_disease_k(model, tokenizer, texts, encoder=None, max_length=64, top_k=3):
    """
    Predict diseases from symptom text.

    Parameters
    ----------
    model : AutoModelForSequenceClassification
    tokenizer : AutoTokenizer
    texts : str or list[str]
    encoder : LabelEncoder (optional)
    max_length : int
    top_k : int

    Returns
    -------
    list[dict]
    """

    if isinstance(texts, str):
        texts = [texts]

    device = next(model.parameters()).device
    model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)

    results = []

    for i, probs in enumerate(probabilities):

        probs_cpu = probs.cpu()

        top_probs, top_indices = torch.topk(probs_cpu, top_k)

        prediction = {}

        prediction["input"] = texts[i]

        prediction["predicted_class_id"] = int(top_indices[0])

        prediction["confidence"] = float(top_probs[0])

        if encoder is not None:
            prediction["predicted_disease"] = encoder.inverse_transform(
                [int(top_indices[0])]
            )[0]
        else:
            prediction["predicted_disease"] = int(top_indices[0])

        prediction["top_predictions"] = []

        for idx, prob in zip(top_indices, top_probs):

            idx = int(idx)
            prob = float(prob)

            if encoder is not None:
                disease = encoder.inverse_transform([idx])[0]
            else:
                disease = idx

            prediction["top_predictions"].append({
                "class_id": idx,
                "disease": disease,
                "probability": prob
            })

        results.append(prediction)

    return results

In [7]:
predict_disease_k(
    model,
    tokenizer,
    [
        " fever and cough and  chest pain",
        "runny nose sore throat",
        "shortness of breath wheezing"
    ],
    encoder
)

[{'input': ' fever and cough and  chest pain',
  'predicted_class_id': 9,
  'confidence': 0.798926055431366,
  'predicted_disease': 'Pneumonia',
  'top_predictions': [{'class_id': 9,
    'disease': 'Pneumonia',
    'probability': 0.798926055431366},
   {'class_id': 3, 'disease': 'Asthma', 'probability': 0.07797181606292725},
   {'class_id': 4,
    'disease': 'Bronchiectasis',
    'probability': 0.04172167554497719}]},
 {'input': 'runny nose sore throat',
  'predicted_class_id': 7,
  'confidence': 0.5828092694282532,
  'predicted_disease': 'Influenza',
  'top_predictions': [{'class_id': 7,
    'disease': 'Influenza',
    'probability': 0.5828092694282532},
   {'class_id': 15,
    'disease': 'bronchitis',
    'probability': 0.23725780844688416},
   {'class_id': 14,
    'disease': 'bronchiolitis',
    'probability': 0.14955736696720123}]},
 {'input': 'shortness of breath wheezing',
  'predicted_class_id': 16,
  'confidence': 0.1928039789199829,
  'predicted_disease': 'chronic obstructive 